<a href="https://colab.research.google.com/github/ronk83952-cmd/General/blob/main/torrent_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent Downloader
Content is saved to Colab Disk

In [8]:
#@title Torrent Class (Local Storage Version) with Progress Bar
!apt-get purge -y python3-libtorrent
!pip install libtorrent termcolor tqdm

import libtorrent as lt
from threading import Thread
import time
import os
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from termcolor import cprint

# Local Colab Paths
tor_path = '/content/torrents/tor/'
save_path = '/content/torrents/downloading/'
completed_path = '/content/torrents/completed/'

for path in [tor_path, save_path, completed_path]:
    if not os.path.exists(path):
        os.makedirs(path)

ses = lt.session()
ses.listen_on(6881, 6891)

class Torrent(Thread):
    state_str = ["Queued", "Checking", "Downloading Metadata", "Downloading", "Finished", "Seeding", "Allocating", "Checking Fastresume"]
    tor_file_status = {}

    def __init__(self):
        Thread.__init__(self)
        self.pbar = widgets.FloatProgress(value=0.0, min=0.0, max=1.0, description='Progress:', bar_style='info', layout=widgets.Layout(width='100%'))
        self.label = widgets.Label(value="Waiting for magnet...")

    def run(self):
        display(widgets.VBox([self.label, self.pbar]))
        self.magnets()

    def magnets(self):
        params = {'save_path': save_path, 'storage_mode': lt.storage_mode_t(2)}
        while True:
            magnet_link = input('Enter Magnet Link (or "exit"): ')
            if magnet_link.lower() == "exit": break
            try:
                handle = lt.add_magnet_uri(ses, magnet_link, params)
                self.tor_file_status[handle] = False
                cprint(f'Magnet link added.', 'green')
            except Exception as e:
                cprint(f'Error: {e}', 'red')

    def check(self):
        for torrent in ses.get_torrents():
            s = torrent.status()
            self.label.value = f"[{self.state_str[s.state]}] {torrent.name()} - Down: {s.download_rate/1000:.1f} kB/s"
            self.pbar.value = s.progress

            if torrent.is_seed():
                torrent.move_storage(completed_path)
                cprint(f'Finished: {torrent.name()}', 'green')
                ses.remove_torrent(torrent)

    def save_tor_file(self, torrent):
        if not torrent.has_metadata(): return False
        t_path = os.path.join(tor_path, torrent.name() + ".torrent")
        if os.path.exists(t_path): return True
        torrent_info = torrent.get_torrent_info()
        torrent_file = lt.create_torrent(torrent_info)
        with open(t_path, "wb") as f: f.write(lt.bencode(torrent_file.generate()))
        return True

    def status_check(self):
        while True:
            self.check()
            if self.tor_file_status:
                for torrent, status in list(self.tor_file_status.items()):
                    if not status: self.tor_file_status[torrent] = self.save_tor_file(torrent)
            time.sleep(2)

t = Torrent()
t.daemon = True
t.start()
t.status_check()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'python3-libtorrent' is not installed, so not removed
The following packages were automatically installed and are no longer required:
  libboost-python1.74.0 libtorrent-rasterbar2.0
Use 'apt autoremove' to remove them.
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


/tmp/ipykernel_3341/3799323806.py:23: DeprecationWarning: listen_on() is deprecated
  ses.listen_on(6881, 6891)


Enter Magnet Link (or "exit"): magnet:?xt=urn:btih:2C7EB24083419B95A788A31BAD2F583566E24A67&dn=Brazzers%20University%20Volume%203%20(Brazzers)%202026%20DVDRip&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337&tr=udp%3A%2F%2Fopen.stealth.si%3A80%2Fannounce&tr=udp%3A%2F%2Ftracker.torrent.eu.org%3A451%2Fannounce&tr=udp%3A%2F%2Ftracker.bittor.pw%3A1337%2Fannounce&tr=udp%3A%2F%2Fpublic.popcorn-tracker.org%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.dler.org%3A6969%2Fannounce&tr=udp%3A%2F%2Fexodus.desync.com%3A6969&tr=udp%3A%2F%2Fopen.demonii.com%3A1337%2Fannounce&tr=udp%3A%2F%2Fglotorrents.pw%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.coppersurfer.tk%3A6969&tr=udp%3A%2F%2Ftorrent.gresille.org%3A80%2Fannounce&tr=udp%3A%2F%2Fp4p.arenabg.com%3A1337&tr=udp%3A%2F%2Ftracker.internetwarriors.net%3A1337


/tmp/ipykernel_3341/3799323806.py:44: DeprecationWarning: add_magnet_uri() is deprecated
  handle = lt.add_magnet_uri(ses, magnet_link, params)


Magnet link added.


/tmp/ipykernel_3341/3799323806.py:53: DeprecationWarning: name() is deprecated
  self.label.value = f"[{self.state_str[s.state]}] {torrent.name()} - Down: {s.download_rate/1000:.1f} kB/s"
/tmp/ipykernel_3341/3799323806.py:56: DeprecationWarning: is_seed() is deprecated
  if torrent.is_seed():
/tmp/ipykernel_3341/3799323806.py:62: DeprecationWarning: has_metadata() is deprecated
  if not torrent.has_metadata(): return False
/tmp/ipykernel_3341/3799323806.py:63: DeprecationWarning: name() is deprecated
  t_path = os.path.join(tor_path, torrent.name() + ".torrent")
/tmp/ipykernel_3341/3799323806.py:65: DeprecationWarning: get_torrent_info() is deprecated
  torrent_info = torrent.get_torrent_info()


Enter Magnet Link (or "exit"): magnet:?xt=urn:btih:2C7EB24083419B95A788A31BAD2F583566E24A67&dn=Brazzers%20University%20Volume%203%20(Brazzers)%202026%20DVDRip&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337&tr=udp%3A%2F%2Fopen.stealth.si%3A80%2Fannounce&tr=udp%3A%2F%2Ftracker.torrent.eu.org%3A451%2Fannounce&tr=udp%3A%2F%2Ftracker.bittor.pw%3A1337%2Fannounce&tr=udp%3A%2F%2Fpublic.popcorn-tracker.org%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.dler.org%3A6969%2Fannounce&tr=udp%3A%2F%2Fexodus.desync.com%3A6969&tr=udp%3A%2F%2Fopen.demonii.com%3A1337%2Fannounce&tr=udp%3A%2F%2Fglotorrents.pw%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.coppersurfer.tk%3A6969&tr=udp%3A%2F%2Ftorrent.gresille.org%3A80%2Fannounce&tr=udp%3A%2F%2Fp4p.arenabg.com%3A1337&tr=udp%3A%2F%2Ftracker.internetwarriors.net%3A1337
Magnet link added.


/tmp/ipykernel_3341/3799323806.py:58: DeprecationWarning: name() is deprecated
  cprint(f'Finished: {torrent.name()}', 'green')


Finished: brazzers-university-3-480p.mp4


KeyboardInterrupt: 

In [19]:
#@title Check Disk Space & Local Files
import os

print("--- Disk Usage ---")
!df -h /content

print("\n--- Completed Downloads ---")
path = '/content/torrents/completed/'
if os.path.exists(path):
    files = os.listdir(path)
    if files:
        for f in files:
            size = os.path.getsize(os.path.join(path, f)) / (1024*1024)
            print(f"{f} ({size:.2f} MB)")
    else:
        print("No files in completed folder.")
else:
    print("Folder not found.")

--- Disk Usage ---
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   45G   69G  40% /

--- Completed Downloads ---
brazzers-university-3-480p.mp4 (1487.58 MB)


### 🎬 Video Player
Select a downloaded file from the dropdown menu and click 'Load Player'.

In [24]:
import os
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import output

# Allow Colab to serve files
output.enable_custom_widget_manager()

def create_player(file_path):
    # Get the relative path for the Colab web server
    rel_path = os.path.relpath(file_path, '/content')
    video_url = f'/view/{rel_path}'

    html_player = f"""
    <div style='max-width: 800px; margin: 10px auto;'>
        <video width='100%' controls autoplay style='border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.5);'>
            <source src='{video_url}' type='video/mp4'>
            Your browser does not support the video tag.
        </video>
        <p style='text-align: center; color: #555;'>Playing: {os.path.basename(file_path)}</p>
    </div>
    """
    display(HTML(html_player))

path = '/content/torrents/completed/'
if os.path.exists(path):
    mp4_files = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.mp4')]

    if mp4_files:
        file_dropdown = widgets.Dropdown(
            options=mp4_files,
            description='File:',
            style={'description_width': 'initial'},
            layout={'width': 'max-content'}
        )
        play_button = widgets.Button(description='Load Player', button_style='primary')

        def on_click(b):
            with output_area:
                output_area.clear_output()
                create_player(file_dropdown.value)

        play_button.on_click(on_click)
        output_area = widgets.Output()

        display(widgets.VBox([file_dropdown, play_button, output_area]))
    else:
        print("No MP4 files found in the completed folder.")
else:
    print("Completed folder not found.")

### 🛠 Advanced Diagnostic & Conversion
If the player above shows an error, use these tools to check the codec or convert the file to a compatible format (H.264).

In [22]:
#@title Check File Details (Codec/Resolution)
import subprocess
import os

def check_file(file_path):
    if not os.path.exists(file_path):
        print(f'❌ File NOT found: {file_path}')
        return

    print(f'✅ File found: {file_path}')
    print(f'📏 Size: {os.path.getsize(file_path) / (1024*1024):.2f} MB')

    try:
        # Get detailed stream info
        cmd = f"ffprobe -v quiet -select_streams v:0 -show_entries stream=codec_name -of default=noprint_wrappers=1:nokey=1 '{file_path}'"
        codec = subprocess.check_output(cmd, shell=True).decode('utf-8').strip()
        print(f'--- Video Codec: {codec} ---')
        if codec == 'hevc':
            print('⚠️ This is an H.265/HEVC file. Most browsers require H.264. Use the conversion cell below.')
        else:
            print('This codec should normally work in most browsers.')
    except Exception as e:
        print(f'Error checking metadata: {e}')

# Automatically check the first file in completed folder
path = '/content/torrents/completed/'
if os.path.exists(path) and os.listdir(path):
    first_file = os.path.join(path, os.listdir(path)[0])
    check_file(first_file)
else:
    print('No files found to check.')

✅ File found: /content/torrents/completed/brazzers-university-3-480p.mp4
📏 Size: 1487.58 MB
--- Video Codec: h264 ---
This codec should normally work in most browsers.


In [23]:
#@title (Optional) Convert to Standard MP4 (H.264)
import os

# If you have a specific file, paste the path here, otherwise it uses the first found file
input_file = "" #@param {type:"string"}
output_file = "/content/torrents/completed/converted_video.mp4"

if not input_file:
    path = '/content/torrents/completed/'
    files = [f for f in os.listdir(path) if f.endswith('.mp4') and f != 'converted_video.mp4']
    if files: input_file = os.path.join(path, files[0])

if input_file and os.path.exists(input_file):
    print(f'Converting {input_file} to standard H.264 MP4...')
    # -preset ultrafast for speed, -c:a copy to keep original audio
    !ffmpeg -i "{input_file}" -c:v libx264 -preset ultrafast -crf 23 -c:a copy "{output_file}" -y
    print(f'\n✅ Done! New file created at: {output_file}')
    print('Refresh the Video Player cell and select "converted_video.mp4" to play.')
else:
    print('No valid input file found.')

Converting /content/torrents/completed/brazzers-university-3-480p.mp4 to standard H.264 MP4...
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --ena

In [25]:
!pip install gradio

In [ ]:
import gradio as gr
import os

def get_video_list():
    path = '/content/torrents/completed/'
    if os.path.exists(path):
        return [os.path.join(path, f) for f in os.listdir(path) if f.endswith(('.mp4', '.mkv', '.webm'))]
    return []

def play_video(file_path):
    return file_path

# Create Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("### 🎬 Gradio Video Player")
    with gr.Row():
        video_dropdown = gr.Dropdown(choices=get_video_list(), label="Select Video File")
        refresh_btn = gr.Button("🔄 Refresh List")

    video_output = gr.Video(label="Player")

    def refresh():
        return gr.Dropdown(choices=get_video_list())

    video_dropdown.change(play_video, inputs=video_dropdown, outputs=video_output)
    refresh_btn.click(refresh, outputs=video_dropdown)

demo.launch(debug=True, inline=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fc6eea41fe2c8726d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
